## Exception Handling Essentials

## Introduction to Exception Handling in Google Cloud Client Libraries

Welcome back! As we continue our exploration of working with Google Cloud services, it's important to focus on building **resilient applications**. A key aspect of resilience is understanding how to handle exceptions that occur when interacting with cloud services.

When using Google Cloud client libraries, exceptions can arise from two main sources: the **service-side** (GCP servers) or the **client-side** (your application environment). Knowing how to identify and respond to these is essential for creating reliable cloud solutions.

---

## Service-Side Exceptions

When a Google Cloud service encounters an error while processing a request, the client library raises a **service-side exception**. These typically indicate that the request was invalid, the resource doesn't exist, or you lack the necessary permissions.



For example, attempting to perform unauthorized operations in Firestore will raise a `PermissionDenied` exception.

```python
from google.cloud import firestore
from google.api_core import exceptions

client = firestore.Client()

try:
    # Attempt to write to a restricted collection
    doc_ref = client.collection('admin-only').document('config')
    doc_ref.set({'restricted_data': True})
except exceptions.PermissionDenied as error:
    print(f"Permission error: {error.message}")
except exceptions.GoogleAPICallError as error:
    print(f"Service-side error: {error.message}")
```

> **Note:** When using `.get()`, Firestore **does not** raise a `NotFound` exception if a document is missing. Instead, it returns a `DocumentSnapshot` where the `exists` property is `False`.

```python
doc_ref = client.collection('users').document('nonexistent-doc')
doc = doc_ref.get()

if not doc.exists:
    print("Document does not exist") # This is normal flow, not an exception
```

---

## Client-Side Exceptions

Exceptions can also occur before the request even reaches Google. These usually involve local network connectivity, invalid configurations, or timeout issues.

Common client-side exceptions include `RetryError` or `InvalidArgument`.

```python
from google.cloud import firestore
from google.api_core import exceptions

try:
    # Invalid endpoint causes a client-side connection issue
    client = firestore.Client(client_options={"api_endpoint": "http://invalid-endpoint:8080"})
    docs = list(client.collection('users').stream())
except exceptions.InvalidArgument as error:
    print(f"Client-side error: {error.message}")
except exceptions.RetryError as error:
    print(f"Retry error: {error.message}")
```

---

## Service-Specific Exceptions Hierarchy

Google Cloud libraries define a hierarchy of exceptions. You can catch specific errors or use a more general parent class like `GoogleAPICallError` to handle a broader range of service issues.

| Exception Class | Description |
| :--- | :--- |
| `PermissionDenied` | Authentication or IAM roles are insufficient. |
| `NotFound` | The requested resource was not found (e.g., a specific bucket or dataset). |
| `DeadlineExceeded` | The request took too long and timed out. |
| `InternalServerError` | A transient error occurred on Google's side. |

---

## Integrating Exception Handling with Retry Strategies

Exception handling is most powerful when combined with **exponential backoff**. This allows your app to handle "transient" errors (like a temporary network blip) automatically.



### Built-in Retry Decorator
```python
from google.api_core import exceptions, retry

# Retries only if a DeadlineExceeded exception occurs
@retry.Retry(predicate=retry.if_exception_type(exceptions.DeadlineExceeded))
def write_with_retry():
    doc_ref = client.collection('users').document('user123')
    doc_ref.set({'name': 'John Doe'})

try:
    write_with_retry()
except exceptions.RetryError:
    print("Operation failed after multiple retries.")
```

---

## Error Handling Best Practices

1.  **Catch Specific Exceptions First**: Always place specific handlers (like `PermissionDenied`) above general ones (like `GoogleAPICallError`).
2.  **Leverage Error Details**: Use `error.message` or `error.code` to provide meaningful feedback to users or logs.
3.  **Differentiate Error Types**: Immediately fail on permanent errors (403 Forbidden), but retry on transient errors (503 Service Unavailable).
4.  **Graceful Recovery**: Implement fallbacks (e.g., show cached data) if the cloud service is unreachable.

## Conclusion

Robust exception handling ensures your cloud application doesn't crash during minor service interruptions. By understanding the GCP exception hierarchy and implementing smart retry logic, you build applications that are truly production-ready.

How would you modify your error handling if you were working with a time-sensitive payment processing service?

## Accessing a Non-Existent Firestore Document: Observing Exception Handling

In this task, your primary objective is to observe a Python script that demonstrates proper exception handling patterns when working with Firestore documents. The script shows how to handle the normal case where a document doesn't exist (which doesn't raise an exception) and implements proper exception handling structure for operations that could potentially fail. Your role is to run the script and understand the error handling patterns - no code modifications are needed for this task.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, bear in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to your clipboard.

```python
from google.cloud import firestore
from google.api_core import exceptions
import uuid

# Create a Firestore client
db = firestore.Client()

# Generate a unique document ID to ensure it doesn't exist
unique_id = str(uuid.uuid4())

# Demonstrate proper exception handling patterns
try:
    # Try to retrieve a document that doesn't exist
    doc_ref = db.collection('users').document(unique_id)
    doc = doc_ref.get()
    
    if doc.exists:
        print(f"Document data: {doc.to_dict()}")
    else:
        print(f"Document does not exist - this is normal behavior")
    
    # Create a different document 
    new_doc_ref = db.collection('users').document(f"user-{unique_id}")
    new_doc_ref.set({'name': 'Test User', 'email': 'test@example.com'})
    print("New document created successfully")

except exceptions.PermissionDenied as e:
    print(f"Permission denied: {e}")

except exceptions.InvalidArgument as e:
    print(f"Invalid argument error: {e}")

except exceptions.GoogleAPICallError as e:
    print(f"Google API error: {e}")
```
This script highlights a fundamental concept in Firestore development: the distinction between **data states** (a document being missing) and **system errors** (a permission failure or network issue).



### Key Observations from the Script

#### 1. "NotFound" is not an Exception for `.get()`
In many database systems, querying for a record that doesn't exist might throw an error. In Firestore, calling `.get()` on a non-existent document is considered a **successful** request. 
*   **The Result:** You receive a `DocumentSnapshot` object.
*   **The Check:** You must use the `.exists` property to verify if data was found.
*   **Why?** This allows your application to handle missing data gracefully without the overhead of a full try/except block for common application logic.

#### 2. Categorizing Service Errors
The `try...except` block is reserved for actual failures in the communication or authorization layer:
*   **`PermissionDenied`**: The most common production error. It triggers if your **IAM roles** or **Firestore Security Rules** block the request.
*   **`InvalidArgument`**: Usually occurs if your document ID contains illegal characters or if your data structure exceeds Firestore limits (e.g., a document larger than 1 MiB).
*   **`GoogleAPICallError`**: A "catch-all" parent class for any other service-side issues (like `503 Service Unavailable`).

### Best Practice Structure
When building professional cloud applications, follow the order shown in the script:
1.  **Check `.exists`** for business logic (does this user have a profile?).
2.  **Catch Specific Exceptions** (Permission, Deadline) to provide clear logs.
3.  **Catch Generic Exceptions** as a safety net.

By separating these concerns, you ensure that your application doesn't crash from a simple missing record, while still remaining protected against critical API failures.

## Google Cloud Firestore Exception Handling

Welcome back! In this task, we will take your understanding of Google Cloud SDK and GCP a notch higher. You are given a Python script that attempts to perform operations on Google Cloud Firestore that can result in various exceptions.

Your task is to enhance this script by incorporating an exception handling mechanism to gracefully handle the exceptions that arise from these operations.

In the exception handling, you should discern between various exception types:

    exceptions.PermissionDenied: In this scenario, print this message: 'You do not have permissions to access the requested resource.'
    exceptions.NotFound: Here, print this message: 'The requested resource does not exist.'

For other exception types, just print this generic message: 'Unexpected error occurred with error: {error}'. You can access the error message directly from the exception object.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top-right corner. While resetting erases any code changes, copying your code to the clipboard preserves it through a reset.

```python
from google.cloud import firestore
from google.api_core import exceptions
import random

# Create a Firestore client
client = firestore.Client()

# Simulate different exception scenarios
try:
    # Randomly simulate different exception types for demonstration
    scenario = random.choice(['permission_denied', 'not_found', 'general_error'])
    
    if scenario == 'permission_denied':
        raise exceptions.PermissionDenied("Simulated permission denied error")
    elif scenario == 'not_found':
        raise exceptions.NotFound("Simulated not found error")
    else:
        raise exceptions.GoogleAPICallError("Simulated general API error")
    
# TODO: Implement error handling for service level issues.

except exceptions.PermissionDenied as e:
    pass

except exceptions.NotFound as e:
    pass

except exceptions.GoogleAPICallError as e:
    pass
```

To enhance the script for resilient cloud engineering, you need to implement a specific hierarchy in your `try...except` block. This ensures that the most specific errors are caught and handled with clear, user-friendly messages before falling back to a generic "catch-all" handler.

Here is the completed script with the requested exception handling logic:

```python
from google.cloud import firestore
from google.api_core import exceptions
import random

# Create a Firestore client
client = firestore.Client()

# Simulate different exception scenarios
try:
    # Randomly simulate different exception types for demonstration
    scenario = random.choice(['permission_denied', 'not_found', 'general_error'])
    
    if scenario == 'permission_denied':
        raise exceptions.PermissionDenied("Simulated permission denied error")
    elif scenario == 'not_found':
        raise exceptions.NotFound("Simulated not found error")
    else:
        raise exceptions.GoogleAPICallError("Simulated general API error")
    
# --- TODO FIXED: Implement error handling for service level issues ---

except exceptions.PermissionDenied:
    print('You do not have permissions to access the requested resource.')

except exceptions.NotFound:
    print('The requested resource does not exist.')

except Exception as error:
    # Catches GoogleAPICallError and any other unexpected issues
    print(f'Unexpected error occurred with error: {error}')
```



### Key Concepts for Professional Exception Handling

*   **Order Matters**: Always catch specific exceptions like `PermissionDenied` and `NotFound` before more general ones like `GoogleAPICallError` or the base `Exception` class. Python checks the `except` blocks from top to bottom.
*   **The `NotFound` Nuance**: While `.get()` on a document returns a snapshot with `exists=False`, other GCP operations (like trying to access a Firestore Database instance that doesn't exist or a Cloud Storage bucket that has been deleted) **will** raise the `exceptions.NotFound` error.
*   **Production Readiness**: In a real application, you might add logging (using the `logging` module) inside these blocks to capture the stack trace for debugging while still showing a polite error message to the end user.

## Enhancing Firestore Script with RetryError Exception Handling

Impressive work! Let's raise the bar again. You're provided with a Python script that tries to retrieve documents from a Google Cloud Firestore collection. The script adeptly handles the GoogleAPICallError exception originating from service's response. This time, your task is to further enhance this script by incorporating handling for the RetryError, a type of exception that symbolizes errors within the Google Cloud client library itself. In case of a RetryError, print this message: 'Unexpected Google Cloud client error: {error_message}'.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.api_core import exceptions
from google.api_core import retry

# Create a Firestore client
client = firestore.Client()

# Attempt to retrieve documents from a collection
try:
    docs = list(client.collection('cosmo-images-archive-2023').stream())
    print(f"Retrieved {len(docs)} documents successfully")
    
    # Simulate a RetryError for demonstration purposes
    raise exceptions.RetryError("Connection failed after maximum retries", [])

# GoogleAPICallError is caught when issues occur at the service level
except exceptions.NotFound as e:
    print("The requested resource does not exist.")

except exceptions.Forbidden as e:
    print("You do not have permissions to access the requested resource.")

except exceptions.GoogleAPICallError as e:
    print(f"Unexpected error occurred with error: {e}")
    
# TODO: Handle possible Google Cloud client library internal errors.

```

Handling a `RetryError` is a critical step in building production-ready cloud applications. While a `GoogleAPICallError` tells you that the **service** (Firestore) had an issue, a `RetryError` tells you that the **client library** gave up after multiple failed attempts. This usually points to persistent network issues or exhausted retry budgets.



Here is the enhanced script with the `RetryError` handling included:

```python
from google.cloud import firestore
from google.api_core import exceptions
from google.api_core import retry

# Create a Firestore client
client = firestore.Client()

# Attempt to retrieve documents from a collection
try:
    docs = list(client.collection('cosmo-images-archive-2023').stream())
    print(f"Retrieved {len(docs)} documents successfully")
    
    # Simulate a RetryError for demonstration purposes
    raise exceptions.RetryError("Connection failed after maximum retries", cause=None)

# GoogleAPICallError is caught when issues occur at the service level
except exceptions.NotFound as e:
    print("The requested resource does not exist.")

except exceptions.Forbidden as e:
    print("You do not have permissions to access the requested resource.")

except exceptions.GoogleAPICallError as e:
    print(f"Unexpected error occurred with error: {e}")
    
# --- TODO FIXED: Handle possible Google Cloud client library internal errors ---
except exceptions.RetryError as e:
    print(f"Unexpected Google Cloud client error: {e}")
```

### Why handling RetryError is unique:

*   **Exhausted Logic**: A `RetryError` specifically means that your **Retry Strategy** (like the one we set up with `total=5` earlier) worked as intended but could not overcome the underlying problem. It acts as a "circuit breaker" to prevent your code from looping infinitely.
*   **The Cause**: Often, a `RetryError` wraps the *last* exception that occurred (like a `DeadlineExceeded`). By catching it, you can log that the system is currently unstable and potentially trigger an alert or a failover mechanism.
*   **User Experience**: Instead of the app hanging while it retries, catching this error allows you to tell the user: "We're having trouble connecting right now. Please check your internet and try again later."